# Physics-Based Validation Layer for Power Grid Operation (Pre-ML)

This notebook implements a physics-based validation layer for synchrophasor power grid
monitoring. The purpose of this layer is to evaluate whether observed grid behavior is
physically plausible based on electrical principles, independent of cyber intent,
attack labels, or operational context.

All runtime features are retained and categorized; however, only measurements with direct
physical interpretation are used to assess physical stress or violations. Absolute phase
angles, control statuses, and cyber-related features are excluded from physical severity
assessment, as they may vary during normal operating conditions such as load changes or
maintenance without indicating unsafe behavior.

Each grid element (generators, transmission lines, and protection relays) is assigned a
continuous health value derived from the maximum physical deviation across its associated
measurements. For interpretability, these health values may be visualized using a
green–amber–red color scheme:

• **GREEN** — Measurements are within normal physical operating limits.  
• **AMBER** — Measurements indicate elevated but physically plausible stress or transient
  deviation.  
• **RED** — Measurements exhibit abnormal physical behavior outside expected operating
  limits.

A red state reflects a physical anomaly only and does not imply malicious intent or attack
activity.

This layer does not perform attack detection or classification. Instead, it provides a
physics-consistent foundation for subsequent machine learning, interpretation, and
operational decision-support layers.


In [31]:
import pandas as pd
import numpy as np
import json

DATA_PATH = "../data/merged/multi_class_dataset_clean_FULL.csv"
df = pd.read_csv(DATA_PATH)

NON_RUNTIME_COLS = ["marker", "marker_name", "label", "label_name"]
FEATURES_ALL = [c for c in df.columns if c not in NON_RUNTIME_COLS]

with open("FEATURE_SET.json", "w") as f:
    json.dump(FEATURES_ALL, f, indent=2)

print("Runtime features:", len(FEATURES_ALL))

Runtime features: 132


In [32]:
FEATURE_TYPES = {}

for col in FEATURES_ALL:

    if ":VH" in col:
        FEATURE_TYPES[col] = "voltage_phase_angle"
        
    elif ":IH" in col:
        FEATURE_TYPES[col] = "current_phase_angle"

    elif ":V" in col and "PM" in col:
        FEATURE_TYPES[col] = "voltage_magnitude"

    elif ":I" in col and "PM" in col:
        FEATURE_TYPES[col] = "current_magnitude"

    elif col.endswith(":DF"):
        FEATURE_TYPES[col] = "rocof"

    elif col.endswith(":F"):
        FEATURE_TYPES[col] = "frequency"

    elif ":Z_inf_flag" in col:
        FEATURE_TYPES[col] = "impedance_flag"
        
    elif "ZH" in col:
        FEATURE_TYPES[col] = "impedance_angle"
    
    elif ":Z" in col:
        FEATURE_TYPES[col] = "impedance"

    elif col.endswith(":S"):
        FEATURE_TYPES[col] = "status"

    elif "log" in col.lower() or "snort" in col.lower():
        FEATURE_TYPES[col] = "cyber_log"

    else:
        FEATURE_TYPES[col] = "other"
        
pd.Series(FEATURE_TYPES).value_counts()

voltage_phase_angle    24
voltage_magnitude      24
current_phase_angle    24
current_magnitude      24
cyber_log              12
frequency               4
rocof                   4
impedance               4
impedance_angle         4
status                  4
impedance_flag          4
Name: count, dtype: int64

In [33]:
# --------------------------------------------------
# Relay / Breaker Status Sanity Check
# --------------------------------------------------

print("=== Relay Status Sanity Check ===")

status_cols = [
    c for c in FEATURES_ALL
    if c.endswith(":S") and c.startswith("R")
]

if not status_cols:
    raise ValueError("❌ No relay status columns found (R*:S)")

print(f"Found {len(status_cols)} relay status columns:")
print(status_cols)
print()

# Check unique values
for col in status_cols:
    vals = sorted(df[col].dropna().unique())
    print(f"{col}: {vals}")

# Check variability across dataset
print("\n=== Status Change Counts ===")
for col in status_cols:
    changes = (df[col].diff() != 0).sum()
    print(f"{col}: {changes} changes")

=== Relay Status Sanity Check ===
Found 4 relay status columns:
['R1:S', 'R2:S', 'R3:S', 'R4:S']

R1:S: [np.int64(0)]
R2:S: [np.int64(0)]
R3:S: [np.int64(0)]
R4:S: [np.int64(0)]

=== Status Change Counts ===
R1:S: 1 changes
R2:S: 1 changes
R3:S: 1 changes
R4:S: 1 changes


In [34]:
# --------------------------------------------------
# Quiet baseline selection and feature statistics
# --------------------------------------------------

print("=== Quiet Baseline Selection ===")

# Define physically quiet operating condition
quiet_mask = (
    (df["marker"] == 41) &
    (df["R1:DF"].abs() < 0.05) &
    (df["R2:DF"].abs() < 0.05) &
    (df["R3:DF"].abs() < 0.05) &
    (df["R4:DF"].abs() < 0.05)
)

# Extract quiet baseline data
df_quiet = df.loc[quiet_mask, FEATURES_ALL]

print(f"Total samples        : {len(df)}")
print(f"Quiet baseline samples: {len(df_quiet)}")
print(f"Quiet ratio           : {len(df_quiet) / len(df):.2%}")
print()

print("=== Computing Feature Statistics ===")

# Compute robust feature statistics
stats = []
skipped_few_samples = 0

for col in FEATURES_ALL:

    # Safety check
    if col not in FEATURE_TYPES:
        continue

    s = df_quiet[col].dropna()

    # Require sufficient samples for stability
    if len(s) < 50:
        skipped_few_samples += 1
        continue

    stats.append({
        "feature": col,
        "type": FEATURE_TYPES[col],
        "median": float(s.median()),
        "q25": float(s.quantile(0.25)),
        "q75": float(s.quantile(0.75)),
        "p95": float(s.quantile(0.95)),
        "p99": float(s.quantile(0.99)),
    })

# Build statistics DataFrame
feature_stats_df = pd.DataFrame(stats).set_index("feature")

print(f"Features with statistics : {len(feature_stats_df)}")
print(f"Features skipped (<50 samples): {skipped_few_samples}")
print()

print("=== Feature Type Breakdown ===")
print(feature_stats_df["type"].value_counts())
print()

# Inspect head
feature_stats_df.head()

# Save calibration statistics
feature_stats_df.to_json(
    "FEATURE_STATS.json",
    orient="index",
    indent=2
)

print("Saved FEATURE_STATS.json")


=== Quiet Baseline Selection ===
Total samples        : 78377
Quiet baseline samples: 4405
Quiet ratio           : 5.62%

=== Computing Feature Statistics ===
Features with statistics : 132
Features skipped (<50 samples): 0

=== Feature Type Breakdown ===
type
voltage_phase_angle    24
voltage_magnitude      24
current_phase_angle    24
current_magnitude      24
cyber_log              12
frequency               4
rocof                   4
impedance               4
impedance_angle         4
status                  4
impedance_flag          4
Name: count, dtype: int64

Saved FEATURE_STATS.json


In [151]:
def get_physical_bounds(row):
    t = row["type"]
    name = str(row.name)

    # --------------------------------------------------
    # Structural zeros (no physical sensor)
    # --------------------------------------------------
    if t == "voltage_magnitude" and any(k in name for k in ["PM8", "PM9"]):
        return {
            "normal": (0.0, 0.0),
            "warning": None,
            "critical": None
        }

    # --------------------------------------------------
    # Voltage magnitude (per-unit, grid code aligned)
    # --------------------------------------------------
    if t == "voltage_magnitude":
        mu = float(row["median"])
        return {
            "normal": (0.95 * mu, 1.05 * mu),
            "warning": (0.90 * mu, 1.10 * mu),
            "critical": (0.85 * mu, 1.15 * mu),
        }

    # --------------------------------------------------
    # Current magnitude (thermal stress model)
    # --------------------------------------------------
    if t == "current_magnitude":
        i90 = max(float(row["p90"]), 1e-6)
        i97 = max(float(row["p97"]), i90 * 1.05)
    
        return {
            "normal":   (0.0, i90),
            "warning":  (i90, i97),
            "critical": (i97, np.inf),
        }
    # --------------------------------------------------
    # Frequency (grid operational limits)
    # --------------------------------------------------
    if t == "frequency":
        return {
            "normal": (59.5, 60.5),
            "warning": (59.0, 61.0),
            "critical": (58.0, 62.0),
        }

    # --------------------------------------------------
    # ROCOF (system stability indicator)
    # --------------------------------------------------
    if t == "rocof":
        return {
            "normal": (-1.0, 1.0),
            "warning": (-2.0, 2.0),
            "critical": (-5.0, 5.0),
        }

    # --------------------------------------------------
    # Impedance infinite flag (PROTECTION DECISION)
    # --------------------------------------------------
    if t == "impedance_flag":
    # Binary physical protection output
        return {
            "normal": (0, 0),
            "warning": None,
            "critical": (1, 1),
        }

    # --------------------------------------------------
    # Angle-based & non-physical signals
    # --------------------------------------------------
    # impedance (raw Z)
    # voltage_phase_angle
    # current_phase_angle
    # impedance_angle
    # status bits, flags, logs
    return None

In [152]:
# --------------------------------------------------
# Physical Bounds
# Apply physical bounds, preview representative R1 bounds, and export
# --------------------------------------------------

print("=== Physical Bounds ===")

# Apply bounds to all features
feature_stats_df["bounds"] = feature_stats_df.apply(
    get_physical_bounds,
    axis=1
)

# Summary
total_features = len(feature_stats_df)
bounded_features = feature_stats_df["bounds"].notna().sum()
ignored_features = total_features - bounded_features

print(f"Total features with stats : {total_features}")
print(f"Physically bounded        : {bounded_features}")
print(f"Ignored (non-physical)    : {ignored_features}")
print()

# --------------------------------------------------
# Preview representative bounds for Relay R1
# --------------------------------------------------

print("=== R1 Physical Bounds (Preview) ===")

r1_bounds_df = feature_stats_df.loc[
    feature_stats_df.index.str.startswith("R1-") &
    feature_stats_df["bounds"].notna(),
    ["type", "bounds"]
].copy()

def unpack_bounds(b):
    return pd.Series({
        "normal": b.get("normal"),
        "warning": b.get("warning"),
        "critical": b.get("critical"),
    })

bounds_expanded = r1_bounds_df["bounds"].apply(unpack_bounds)

r1_bounds_table = pd.concat(
    [r1_bounds_df.drop(columns=["bounds"]), bounds_expanded],
    axis=1
)

print(f"Number of R1 physical features: {len(r1_bounds_table)}")
display(r1_bounds_table.sort_index())
print()

# --------------------------------------------------
# Export physical bounds (JSON-safe)
# --------------------------------------------------

physical_bounds_json = {}

for feature, row in feature_stats_df.iterrows():
    bounds = row["bounds"]

    if bounds is None:
        physical_bounds_json[feature] = {
            "type": row["type"],
            "bounds": None
        }
    else:
        physical_bounds_json[feature] = {
            "type": row["type"],
            "bounds": {
                k: list(v) if v is not None else None
                for k, v in bounds.items()
            }
        }

with open("PHYSICAL_FEATURE_BOUNDS.json", "w") as f:
    json.dump(physical_bounds_json, f, indent=2)

print("Saved PHYSICAL_FEATURE_BOUNDS.json")


=== Physical Bounds ===


KeyError: 'p90'

In [153]:
# --------------------------------------------------
# Feature-level physical severity computation (FINAL, CORRECT)
# --------------------------------------------------

print("=== Feature-Level Physical Severity ===")

def feature_severity(value, bounds):
    """
    Returns:
      0.0 = GREEN
      0.6 = AMBER
      1.0 = RED

    Supports:
      - Continuous physical signals (V, I, F, ROCOF, Z)
      - Discrete protection flags (impedance_flag)
    """
    if bounds is None or pd.isna(value):
        return 0.0

    # --------------------------------------------------
    # DISCRETE FLAG (impedance_flag)
    # --------------------------------------------------
    # Identified by having no warning band
    if bounds.get("warning") is None and bounds.get("critical") is not None:
        crit_lo, crit_hi = bounds["critical"]
        return 1.0 if crit_lo <= value <= crit_hi else 0.0

    # --------------------------------------------------
    # CONTINUOUS SIGNALS
    # --------------------------------------------------
    lo, hi = bounds["normal"]
    if lo <= value <= hi:
        return 0.0

    lo, hi = bounds["warning"]
    if lo <= value <= hi:
        return 0.6

    lo, hi = bounds["critical"]
    if value < lo or value > hi:
        return 1.0

    return 0.0


# --------------------------------------------------
# Build severity DataFrame
# --------------------------------------------------

severity_df = pd.DataFrame(
    {
        feature: df[feature].apply(
            lambda v: feature_severity(v, row["bounds"])
        )
        for feature, row in feature_stats_df.iterrows()
    },
    index=df.index
)

print(f"Severity matrix shape: {severity_df.shape}")
print("\nSeverity value counts:")
print(pd.Series(severity_df.values.ravel()).value_counts())

=== Feature-Level Physical Severity ===
Severity matrix shape: (78377, 132)

Severity value counts:
0.0    10281620
0.6       53238
1.0       10906
Name: count, dtype: int64


In [154]:
# --------------------------------------------------
# NORMAL operation validation (marker = 41)
# --------------------------------------------------

NORMAL_MARKER = 41
normal_idx = df[df["marker"] == NORMAL_MARKER].index

print("=== NORMAL OPERATION TEST ===")
print(f"Normal samples: {len(normal_idx)}")
print()

# -----------------------------
# Feature-level checks
# -----------------------------
critical_features = (severity_df.loc[normal_idx] == 1.0).sum().sum()
warning_features  = (severity_df.loc[normal_idx] == 0.6).sum().sum()

print("Feature-level severity:")
print(f"  CRITICAL count : {critical_features}")
print(f"  WARNING count  : {warning_features}")
print()

=== NORMAL OPERATION TEST ===
Normal samples: 4405

Feature-level severity:
  CRITICAL count : 0
  WARNING count  : 720



In [155]:
import numpy as np
import pandas as pd

# ==================================================
# 1. Helper functions
# ==================================================

def current_cols(cols):
    """Current magnitude only (exclude angles)."""
    return [c for c in cols if FEATURE_TYPES.get(c) == "current_magnitude"]

def flag_cols(cols):
    """Impedance protection flags."""
    return [c for c in cols if FEATURE_TYPES.get(c) == "impedance_flag"]

def cols_for_relays(relays, cols):
    """Support R1:XXX and R1-XXX naming."""
    return [
        c for c in cols
        if any(c.startswith(r + ":") or c.startswith(r + "-") for r in relays)
    ]


# ==================================================
# 2. RAW Relay Health (NO CLAMPING)
# ==================================================

PHYSICAL_TYPES = {
    "voltage_magnitude",
    "current_magnitude",
    "frequency",
    "rocof",
    "impedance_flag",
}

relay_health_raw = {}

for relay in ["R1", "R2", "R3", "R4"]:
    cols = [
        c for c in severity_df.columns
        if (c.startswith(relay + ":") or c.startswith(relay + "-"))
        and FEATURE_TYPES.get(c) in PHYSICAL_TYPES
    ]

    if not cols:
        relay_health_raw[relay] = 0.0
        continue

    row = severity_df[cols]

    relay_health_raw[relay] = np.where(
        (row == 1.0).any(axis=1), 1.0,
        np.where((row == 0.6).any(axis=1), 0.6, 0.0)
    )

relay_health_raw = pd.DataFrame(relay_health_raw, index=df.index)


# ==================================================
# 3. LINE HEALTH (FLAG → RED, CURRENT → AMBER)
# ==================================================

L1_ZONE = cols_for_relays(["R1", "R2"], severity_df.columns)
L2_ZONE = cols_for_relays(["R3", "R4"], severity_df.columns)

L1_FLAGS = flag_cols(L1_ZONE)
L2_FLAGS = flag_cols(L2_ZONE)

L1_I = current_cols(L1_ZONE)
L2_I = current_cols(L2_ZONE)

def line_health_physical(sev_row, flags, currents, other_flags):
    """
    Physical line inference (PRIMARY + BACKUP):
    - RED   : impedance flag OR dual-ended critical current
    - AMBER : single-ended current stress
    - GREEN : otherwise
    """

    # -------------------------
    # PRIMARY: impedance trip
    # -------------------------
    if flags and (sev_row[flags] == 1.0).any():
        return 1.0

    # -------------------------
    # BLOCK if other line tripped
    # -------------------------
    if other_flags and (sev_row[other_flags] == 1.0).any():
        return 0.0

    # -------------------------
    # BACKUP: dual-ended overcurrent
    # -------------------------
    if currents:
        crit_curr = (sev_row[currents] == 1.0).sum()
        warn_curr = (sev_row[currents] >= 0.6).sum()

        # both relays see critical current → fault
        if crit_curr >= 1:
            return 1.0

        # one relay sees stress → amber
        if warn_curr >= 1:
            return 0.6

    return 0.0

line_health = pd.DataFrame(index=df.index)
line_health["L1"] = severity_df.apply(
    lambda r: line_health_physical(r, L1_FLAGS, L1_I, L2_FLAGS),
    axis=1
)

line_health["L2"] = severity_df.apply(
    lambda r: line_health_physical(r, L2_FLAGS, L2_I, L1_FLAGS),
    axis=1
)

# ---- Mutual exclusion (safety net only) ----
both_active = (line_health["L1"] > 0) & (line_health["L2"] > 0)

score_L1 = (severity_df[L1_FLAGS + L1_I] > 0).sum(axis=1) if (L1_FLAGS or L1_I) else 0
score_L2 = (severity_df[L2_FLAGS + L2_I] > 0).sum(axis=1) if (L2_FLAGS or L2_I) else 0

line_health.loc[both_active & (score_L1 >= score_L2), "L2"] = 0.0
line_health.loc[both_active & (score_L2 >  score_L1), "L1"] = 0.0


# ==================================================
# 4. Clamp Relays to Their Line (DISPLAY CONSISTENCY)
# ==================================================

relay_health = relay_health_raw.copy()

RELAY_TO_LINE = {
    "R1": "L1", "R2": "L1",
    "R3": "L2", "R4": "L2",
}

for relay, line in RELAY_TO_LINE.items():
    relay_health[relay] = np.minimum(relay_health[relay], line_health[line])


# ==================================================
# 5. Generator Health (Frequency + ROCOF)
# ==================================================

gen_health = pd.DataFrame(index=df.index)

for gen, cols in {
    "G1": ["R1:F", "R1:DF"],
    "G2": ["R3:F", "R3:DF"],
}.items():

    valid = [c for c in cols if c in severity_df.columns]

    if not valid:
        gen_health[gen] = 0.0
        continue

    row = severity_df[valid]

    gen_health[gen] = np.where(
        (row == 1.0).any(axis=1), 1.0,
        np.where((row == 0.6).any(axis=1), 0.6, 0.0)
    )


# ==================================================
# 6. Assemble FINAL Element Health Table
# ==================================================

element_health_final = pd.concat(
    [gen_health, relay_health, line_health],
    axis=1
)

for x in ["B1", "B2", "B3", "BR1", "BR2", "BR3", "BR4"]:
    element_health_final[x] = 0.0

print("=== FINAL Element Physical Health (FIXED) ===")
display(element_health_final.head())

=== FINAL Element Physical Health (FIXED) ===


,G1,G2,R1,R2,R3,R4,L1,L2,B1,B2,B3,BR1,BR2,BR3,BR4
0,0.0,0.0,0.6,0.6,0.0,0.0,0.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.6,0.6,0.0,0.0,0.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.6,0.6,0.0,0.0,0.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [243]:
import numpy as np
import pandas as pd

# ==================================================
# CONFIG
# ==================================================

CURRENT_STRESS_THRESHOLD = 3   # << interpretable knob

RELAYS = ["R1", "R2", "R3", "R4"]
LINES = {"L1": ["R1","R2"], "L2": ["R3","R4"]}

# ==================================================
# HELPERS
# ==================================================

def cols_for(relays, cols):
    return [c for c in cols if any(c.startswith(r + ":") or c.startswith(r + "-") for r in relays)]

def current_cols(cols):
    return [c for c in cols if FEATURE_TYPES.get(c) == "current_magnitude"]

def flag_cols(cols):
    return [c for c in cols if FEATURE_TYPES.get(c) == "impedance_flag"]

# ==================================================
# RELAY HEALTH (independent, interpretable)
# ==================================================

relay_health = {}

for r in RELAYS:
    r_cols = cols_for([r], severity_df.columns)
    flags = flag_cols(r_cols)
    currents = current_cols(r_cols)

    def relay_state(row):
        if flags and (row[flags] == 1.0).any():
            return 1.0
        if currents and (row[currents] >= 0.6).any():
            return 0.6
        return 0.0

    relay_health[r] = severity_df.apply(relay_state, axis=1)

relay_health = pd.DataFrame(relay_health, index=df.index)

# ==================================================
# LINE HEALTH (MERGED: protection + localized stress)
# ==================================================

line_health = {}

for line, rels in LINES.items():
    zone = cols_for(rels, severity_df.columns)
    flags = flag_cols(zone)
    currents = current_cols(zone)

    # other line (for blocking / comparison)
    other_line = [l for l in LINES if l != line][0]
    other_zone = cols_for(LINES[other_line], severity_df.columns)
    other_flags = flag_cols(other_zone)
    other_currents = current_cols(other_zone)

    def line_state(row):
        # ----------------------------------
        # 1) TRUE protection decision
        # ----------------------------------
        if flags and (row[flags] == 1.0).any():
            return 1.0

        # ----------------------------------
        # 2) BLOCK if other line is faulted
        # ----------------------------------
        if other_flags and (row[other_flags] == 1.0).any():
            return 0.0

        # ----------------------------------
        # 3) Localized current stress (soft)
        # ----------------------------------
        own_stress = (row[currents] >= 0.6).sum() if currents else 0
        other_stress = (row[other_currents] >= 0.6).sum() if other_currents else 0

        if (
            own_stress >= CURRENT_STRESS_THRESHOLD
            and own_stress > other_stress
        ):
            return 0.6

        return 0.0

    line_health[line] = severity_df.apply(line_state, axis=1)

line_health = pd.DataFrame(line_health, index=df.index)

# ==================================================
# FIX COLOURING: Clamp relays by their line
# ==================================================

RELAY_TO_LINE = {
    "R1": "L1", "R2": "L1",
    "R3": "L2", "R4": "L2",
}

for relay, line in RELAY_TO_LINE.items():
    relay_health[relay] = np.minimum(
        relay_health[relay],
        line_health[line]
    )

# ==================================================
# GENERATOR HEALTH
# ==================================================

gen_health = {}

for g, cols in {"G1":["R1:F","R1:DF"], "G2":["R3:F","R3:DF"]}.items():
    valid = [c for c in cols if c in severity_df.columns]
    if not valid:
        gen_health[g] = 0.0
        continue

    gen_health[g] = np.where(
        (severity_df[valid] == 1.0).any(axis=1), 1.0,
        np.where((severity_df[valid] == 0.6).any(axis=1), 0.6, 0.0)
    )

gen_health = pd.DataFrame(gen_health, index=df.index)

# ==================================================
# FINAL PHYSICAL LAYER
# ==================================================

element_health_final = pd.concat(
    [gen_health, relay_health, line_health],
    axis=1
)

# Non-physical elements
for x in ["B1","B2","B3","BR1","BR2","BR3","BR4"]:
    element_health_final[x] = 0.0

order = ["G1","G2","R1","R2","R3","R4","L1","L2","B1","B2","B3","BR1","BR2","BR3","BR4"]
element_health_final = element_health_final[order]

print("=== FINAL Element Physical Health (MY WAY) ===")
display(element_health_final.head())

=== FINAL Element Physical Health (MY WAY) ===


,G1,G2,R1,R2,R3,R4,L1,L2,B1,B2,B3,BR1,BR2,BR3,BR4
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [246]:
# ==========================================
# DIAGNOSTIC: Current severity distribution
# ==========================================

for line, currents in {
    "L1": L1_I,
    "L2": L2_I
}.items():
    if not currents:
        continue

    print(f"\n=== {line} CURRENT SEVERITY ===")
    print(
        severity_df[currents]
        .apply(lambda r: (r >= 0.6).sum(), axis=1)
        .value_counts()
        .sort_index()
        .head(10)
    )


=== L1 CURRENT SEVERITY ===
0    74303
1      250
2      209
3      220
4       56
5      213
6      328
7      110
8     2688
Name: count, dtype: int64

=== L2 CURRENT SEVERITY ===
0    74347
1      270
2      215
3      173
4       97
5      199
6      368
7       95
8     2613
Name: count, dtype: int64


In [247]:
# --------------------------------------------------
# TEST 1: Normal operation sanity
# --------------------------------------------------

normal_idx = df[df["marker"] == 41].index

print("=== NORMAL OPERATION TEST ===")
print(f"Normal samples: {len(normal_idx)}")

print("\nElement-level RED count (should be 0):")
print((element_health_final.loc[normal_idx] == 1.0).sum())

print("\nElement-level AMBER count (allowed, should be small):")
print((element_health_final.loc[normal_idx] == 0.6).sum())

=== NORMAL OPERATION TEST ===
Normal samples: 4405

Element-level RED count (should be 0):
G1     0
G2     0
R1     0
R2     0
R3     0
R4     0
L1     0
L2     0
B1     0
B2     0
B3     0
BR1    0
BR2    0
BR3    0
BR4    0
dtype: int64

Element-level AMBER count (allowed, should be small):
G1     0
G2     0
R1     1
R2     1
R3     1
R4     1
L1     1
L2     1
B1     0
B2     0
B3     0
BR1    0
BR2    0
BR3    0
BR4    0
dtype: int64


In [248]:
# --------------------------------------------------
# TEST 2: Line activation by scenario
# --------------------------------------------------

SCENARIO_LINE_MAP = {
    1: "L1", 2: "L1", 3: "L1",
    4: "L2", 5: "L2", 6: "L2",
    7: "L1", 8: "L1", 9: "L1",
    10: "L2", 11: "L2", 12: "L2",
}

for marker, line in SCENARIO_LINE_MAP.items():
    idx = df[df["marker"] == marker].index
    if len(idx) == 0:
        continue

    active = (element_health_final.loc[idx, line] > 0).mean()
    other = "L2" if line == "L1" else "L1"
    inactive = (element_health_final.loc[idx, other] > 0).mean()

    print(f"Marker {marker}:")
    print(f"  {line} active ratio    : {active:.2%}")
    print(f"  {other} active ratio   : {inactive:.2%}")

Marker 1:
  L1 active ratio    : 16.37%
  L2 active ratio   : 0.61%
Marker 2:
  L1 active ratio    : 13.89%
  L2 active ratio   : 0.44%
Marker 3:
  L1 active ratio    : 14.86%
  L2 active ratio   : 0.60%
Marker 4:
  L2 active ratio    : 10.24%
  L1 active ratio   : 0.27%
Marker 5:
  L2 active ratio    : 8.43%
  L1 active ratio   : 0.32%
Marker 6:
  L2 active ratio    : 11.02%
  L1 active ratio   : 1.06%
Marker 7:
  L1 active ratio    : 11.74%
  L2 active ratio   : 2.88%
Marker 8:
  L1 active ratio    : 12.89%
  L2 active ratio   : 2.31%
Marker 9:
  L1 active ratio    : 11.98%
  L2 active ratio   : 1.89%
Marker 10:
  L2 active ratio    : 8.04%
  L1 active ratio   : 0.17%
Marker 11:
  L2 active ratio    : 9.29%
  L1 active ratio   : 0.21%
Marker 12:
  L2 active ratio    : 7.64%
  L1 active ratio   : 0.56%


In [249]:
# --------------------------------------------------
# TEST 3: Relay–Line consistency
# --------------------------------------------------

violations = 0

RELAY_TO_LINE = {"R1": "L1", "R2": "L1", "R3": "L2", "R4": "L2"}

for relay, line in RELAY_TO_LINE.items():
    bad = (element_health_final[relay] == 1.0) & (element_health_final[line] == 0.0)
    violations += bad.sum()

print("=== RELAY–LINE CONSISTENCY ===")
print(f"Violations (should be 0): {violations}")

=== RELAY–LINE CONSISTENCY ===
Violations (should be 0): 0


In [250]:
# --------------------------------------------------
# TEST 4: Protection flag effectiveness
# --------------------------------------------------

flag_cols = [c for c in severity_df.columns if FEATURE_TYPES.get(c) == "impedance_flag"]

print("Flag columns:", flag_cols)

for col in flag_cols:
    triggered = (df[col] == 1).sum()
    if triggered == 0:
        continue

    # Which line does this flag belong to?
    if col.startswith(("R1", "R2")):
        line = "L1"
    else:
        line = "L2"

    ratio = (element_health_final.loc[df[col] == 1, line] == 1.0).mean()

    print(f"{col}:")
    print(f"  Flag activations      : {triggered}")
    print(f"  Line RED ratio        : {ratio:.2%}")

Flag columns: ['R1-PA:Z_inf_flag', 'R2-PA:Z_inf_flag', 'R3-PA:Z_inf_flag', 'R4-PA:Z_inf_flag']
R1-PA:Z_inf_flag:
  Flag activations      : 2877
  Line RED ratio        : 100.00%
R2-PA:Z_inf_flag:
  Flag activations      : 2607
  Line RED ratio        : 100.00%
R3-PA:Z_inf_flag:
  Flag activations      : 2633
  Line RED ratio        : 100.00%
R4-PA:Z_inf_flag:
  Flag activations      : 2789
  Line RED ratio        : 100.00%


In [251]:
def inspect_sample(idx):
    print("=" * 60)
    print(f"Sample index: {idx}")
    print(f"Marker: {df.loc[idx, 'marker']}")
    print("=" * 60)

    # -----------------------------
    # Impedance flags (PHYSICAL)
    # -----------------------------
    print("\n--- Impedance flags (severity) ---")
    found = False
    for c in severity_df.columns:
        if FEATURE_TYPES.get(c) == "impedance_flag" and severity_df.loc[idx, c] == 1.0:
            print(f"  {c} = 1")
            found = True
    if not found:
        print("  (none)")

    # -----------------------------
    # Line health
    # -----------------------------
    print("\n--- Line health ---")
    display(
        element_health_final.loc[idx, ["L1", "L2"]]
        .to_frame("health")
    )

    # -----------------------------
    # Relay health
    # -----------------------------
    print("\n--- Relay health ---")
    display(
        element_health_final.loc[idx, ["R1", "R2", "R3", "R4"]]
        .to_frame("health")
    )

In [252]:
inspect_sample(df[df["marker"] == 1].index[5])
inspect_sample(df[df["marker"] == 2].index[5])
inspect_sample(df[df["marker"] == 3].index[5])
inspect_sample(df[df["marker"] == 4].index[5])
inspect_sample(df[df["marker"] == 5].index[5])
inspect_sample(df[df["marker"] == 6].index[5])
inspect_sample(df[df["marker"] == 7].index[5])
inspect_sample(df[df["marker"] == 8].index[5])

Sample index: 5031
Marker: 1

--- Impedance flags (severity) ---
  R1-PA:Z_inf_flag = 1
  R2-PA:Z_inf_flag = 1

--- Line health ---


,health
L1,1.0
L2,0.0



--- Relay health ---


,health
R1,1.0
R2,1.0
R3,0.0
R4,0.0


Sample index: 4877
Marker: 2

--- Impedance flags (severity) ---
  (none)

--- Line health ---


...

,health
R1,0.0
R2,0.0
R3,0.0
R4,0.0


Sample index: 3919
Marker: 8

--- Impedance flags (severity) ---
  (none)

--- Line health ---


,health
L1,0.0
L2,0.0



--- Relay health ---


,health
R1,0.0
R2,0.0
R3,0.0
R4,0.0


In [226]:
# pick a sample where L1 is actually active
fault_idx = (
    element_health_final
    .query("L2 > 0")
    .index[3]
)

inspect_sample(fault_idx)

Sample index: 861
Marker: 36

--- Impedance flags (severity) ---
  R4-PA:Z_inf_flag = 1

--- Line health ---


,health
L1,0.0
L2,1.0



--- Relay health ---


,health
R1,0.0
R2,0.0
R3,0.0
R4,1.0


In [116]:
def test_sample(idx, df, element_health_final):
    print("=" * 60)
    print(f"Sample index: {idx}")
    print(f"Marker: {df.loc[idx, 'marker']}")
    print("=" * 60)

    row = element_health_final.loc[idx]

    for elem, val in row.items():
        if val == 1.0:
            state = "RED"
        elif val == 0.6:
            state = "AMBER"
        else:
            state = "GREEN"
        print(f"{elem:>4} : {state}")

In [60]:
# Normal
idx = df[df["marker"] == 41].index[3]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 1].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 2].index[0]
test_sample(idx, df, element_health_final)

# L1 severe fault (marker 3)
idx = df[df["marker"] == 3].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 4].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 5].index[0]
test_sample(idx, df, element_health_final)

# L2 severe fault (marker 6)
idx = df[df["marker"] == 6].index[0]
test_sample(idx, df, element_health_final)

# L2 severe fault (marker 6)
idx = df[df["marker"] == 7].index[0]
test_sample(idx, df, element_health_final)

# L2 severe fault (marker 6)
idx = df[df["marker"] == 8].index[0]
test_sample(idx, df, element_health_final)

# L2 severe fault (marker 6)
idx = df[df["marker"] == 9].index[0]
test_sample(idx, df, element_health_final)

# L2 severe fault (marker 6)
idx = df[df["marker"] == 10].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 11].index[10]
test_sample(idx, df, element_health_final)

# L1 severe fault (marker 3)
idx = df[df["marker"] == 12].index[0]
test_sample(idx, df, element_health_final)

# L2 severe fault (marker 6)
idx = df[df["marker"] == 13].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 14].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 15].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 16].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 17].index[0]
test_sample(idx, df, element_health_final)

# L2 severe fault (marker 6)
idx = df[df["marker"] == 18].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 19].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 20].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 21].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 22].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 23].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 24].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 25].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 26].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 27].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 28].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 29].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 30].index[0]
test_sample(idx, df, element_health_final)

idx = df[df["marker"] == 31].index[0]
test_sample(idx, df, element_health_final)
# Check if any RED in normal
normal_idx = df[df["marker"] == 41].index
print("\nRED elements in normal (should be 0):",
      (element_health_final.loc[normal_idx].max(axis=1) == 1.0).sum())

Sample index: 3
Marker: 41
  G1 : GREEN
  G2 : GREEN
  R1 : GREEN
  R2 : GREEN
  R3 : GREEN
  R4 : GREEN
  L1 : GREEN
  L2 : GREEN
  B1 : GREEN
  B2 : GREEN
  B3 : GREEN
 BR1 : GREEN
 BR2 : GREEN
 BR3 : GREEN
 BR4 : GREEN
Sample index: 5026
Marker: 1
  G1 : GREEN
  G2 : GREEN
  R1 : GREEN
  R2 : GREEN
  R3 : GREEN
  R4 : GREEN
  L1 : GREEN
  L2 : GREEN
  B1 : GREEN
  B2 : GREEN
  B3 : GREEN
 BR1 : GREEN
 BR2 : GREEN
 BR3 : GREEN
 BR4 : GREEN
Sample index: 4872
Marker: 2
  G1 : GREEN
  G2 : GREEN
  R1 : GREEN
  R2 : GREEN
  R3 : GREEN
  R4 : GREEN
  L1 : GREEN
  L2 : GREEN
  B1 : GREEN
  B2 : GREEN
  B3 : GREEN
 BR1 : GREEN
 BR2 : GREEN
 BR3 : GREEN
 BR4 : GREEN
Sample index: 4734
Marker: 3
  G1 : GREEN
  G2 : GREEN
  R1 : GREEN
  R2 : GREEN
  R3 : GREEN
  R4 : GREEN
  L1 : GREEN
  L2 : GREEN
  B1 : GREEN
  B2 : GREEN
  B3 : GREEN
 BR1 : GREEN
 BR2 : GREEN
 BR3 : GREEN
 BR4 : GREEN
Sample index: 4591
Marker: 4
  G1 : GREEN
  G2 : GREEN
  R1 : GREEN
  R2 : GREEN
  R3 : GREEN
  R4 : GREEN

IndexError: index 0 is out of bounds for axis 0 with size 0

In [203]:
def inspect_sample(idx, df, severity_df, element_health_df):
    print("=" * 60)
    print(f"Sample index: {idx}")
    print(f"Marker: {df.loc[idx, 'marker']}")
    print("=" * 60)

    # -----------------------------
    # Feature-level severity
    # -----------------------------
    sev_row = severity_df.loc[idx]
    abnormal = sev_row[sev_row > 0].sort_values(ascending=False)

    print("\n--- Feature-level physical severity (non-GREEN) ---")
    if abnormal.empty:
        print("All features GREEN")
    else:
        display(
            abnormal.to_frame("severity")
            .assign(color=abnormal.map(sev_to_color))
        )

    # -----------------------------
    # Element-level health
    # -----------------------------
    elem_row = element_health_df.loc[idx]

    print("\n--- Element-level physical health ---")
    display(
        elem_row.to_frame("health")
        .assign(color=elem_row.map(sev_to_color))
    )

In [127]:
# Pick a NORMAL sample
idx_normal = df[df["marker"] == 41].index[0]

inspect_sample(
    idx_normal,
    df,
    severity_df,
    element_health_df
)

Sample index: 0
Marker: 41

--- Feature-level physical severity (non-GREEN) ---


,severity,color
R1-PM4:I,0.6,AMBER
R1-PM5:I,0.6,AMBER
R1-PM6:I,0.6,AMBER
R1-PM10:I,0.6,AMBER
R2-PM4:I,0.6,AMBER
R2-PM5:I,0.6,AMBER
R2-PM6:I,0.6,AMBER
R2-PM10:I,0.6,AMBER
R3-PM4:I,0.6,AMBER
R3-PM5:I,0.6,AMBER



--- Element-level physical health ---


,health,color
R1,0.0,GREEN
R2,0.0,GREEN
R3,0.0,GREEN
R4,0.0,GREEN
L1,0.0,GREEN
L2,0.0,GREEN
G1,0.0,GREEN
G2,0.0,GREEN
B1,0.0,GREEN
B2,0.0,GREEN


In [128]:
# Example: Fault 80–90% on L1 (marker = 3)
idx_fault = df[df["marker"] == 3].index[0]

inspect_sample(
    idx_fault,
    df,
    severity_df,
    element_health_df
)

Sample index: 4734
Marker: 3

--- Feature-level physical severity (non-GREEN) ---


,severity,color
R1-PA:Z,1.0,RED
R2-PA:Z,1.0,RED
R3-PA:Z,1.0,RED
R4-PA:Z,1.0,RED



--- Element-level physical health ---


,health,color
R1,1.0,RED
R2,1.0,RED
R3,1.0,RED
R4,1.0,RED
L1,0.0,GREEN
L2,0.0,GREEN
G1,0.0,GREEN
G2,0.0,GREEN
B1,0.0,GREEN
B2,0.0,GREEN


In [103]:
normal_idx = df[df["marker"] == 41].index
print("Max element health during normal:")
print(element_health_df.loc[normal_idx].max().sort_values(ascending=False))

Max element health during normal:
R1     1.0
R2     1.0
R3     1.0
R4     1.0
L1     0.0
L2     0.0
G1     0.0
G2     0.0
B1     0.0
B2     0.0
B3     0.0
BR1    0.0
BR2    0.0
BR3    0.0
BR4    0.0
dtype: float64


In [104]:
# Normal operation
print((severity_df.loc[df["marker"] == 41] == 1.0).sum().sum())

12


In [124]:
element_health_df.loc[df["marker"] == 8].max()

R1     1.0
R2     1.0
R3     1.0
R4     1.0
L1     0.0
L2     0.0
G1     0.0
G2     0.0
B1     0.0
B2     0.0
B3     0.0
BR1    0.0
BR2    0.0
BR3    0.0
BR4    0.0
dtype: float64

In [123]:

idx = df[df["marker"] == 20].index[0]

print("Max severity in this sample:")
print(severity_df.loc[idx])


Max severity in this sample:
R1-PA1:VH           0.0
R1-PM1:V            0.0
R1-PA2:VH           0.0
R1-PM2:V            0.0
R1-PA3:VH           0.0
                   ... 
snort_log4          0.0
R1-PA:Z_inf_flag    0.0
R2-PA:Z_inf_flag    0.0
R3-PA:Z_inf_flag    0.0
R4-PA:Z_inf_flag    0.0
Name: 2568, Length: 132, dtype: float64


In [344]:
# --------------------------------------------------
# Compute physical severity matrix (FULL DATASET)
# --------------------------------------------------

def compute_severity_column(col):
    """
    Compute severity time series for one feature column
    using precomputed physical bounds.
    """
    bounds = feature_stats_df.loc[col, "bounds"]

    # Features ignored at physical layer → zero severity
    if bounds is None:
        return pd.Series(0.0, index=df.index)

    return df[col].apply(lambda v: feature_severity(v, bounds))


# Build severity DataFrame (one column per feature)
severity_df = pd.concat(
    {col: compute_severity_column(col) for col in feature_stats_df.index},
    axis=1
)

# Defragment memory (important for later groupby / slicing)
severity_df = severity_df.copy()

print("✅ Severity matrix built")
print("Shape:", severity_df.shape)

display(severity_df.head(30))

✅ Severity matrix built
Shape: (78377, 132)


,R1-PA1:VH,R1-PM1:V,R1-PA2:VH,R1-PM2:V,R1-PA3:VH,R1-PM3:V,R1-PA4:IH,R1-PM4:I,R1-PA5:IH,R1-PM5:I,...,relay3_log,relay4_log,snort_log1,snort_log2,snort_log3,snort_log4,R1-PA:Z_inf_flag,R2-PA:Z_inf_flag,R3-PA:Z_inf_flag,R4-PA:Z_inf_flag
0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.6,0.0,0.6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.6,0.0,0.6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.6,0.0,0.6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.0,0.6,0.0,0.6,0.0,0.6,0.0,0.6,0.0,0.6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [345]:
# VH severity in normal operation
severity_df[df["marker"] == 41][
    [c for c in severity_df.columns if ":VH" in c]
].max()

R1-PA1:VH    0.6
R1-PA2:VH    0.0
R1-PA3:VH    0.0
R1-PA7:VH    0.6
R1-PA8:VH    0.0
R1-PA9:VH    0.0
R2-PA1:VH    0.6
R2-PA2:VH    0.0
R2-PA3:VH    0.0
R2-PA7:VH    0.6
R2-PA8:VH    0.0
R2-PA9:VH    0.0
R3-PA1:VH    0.6
R3-PA2:VH    0.0
R3-PA3:VH    0.0
R3-PA7:VH    0.6
R3-PA8:VH    0.0
R3-PA9:VH    0.0
R4-PA1:VH    0.6
R4-PA2:VH    0.0
R4-PA3:VH    0.0
R4-PA7:VH    0.6
R4-PA8:VH    0.0
R4-PA9:VH    0.0
dtype: float64

In [310]:
# Check status distribution in normal operation
status_cols = [c for c in df.columns if c.endswith(":S")]
print(df.loc[df["marker"]==41, status_cols].value_counts().head(10))
print(df.loc[df["marker"]==41, status_cols].mean())

R1:S  R2:S  R3:S  R4:S
0     0     0     0       4405
Name: count, dtype: int64
R1:S    0.0
R2:S    0.0
R3:S    0.0
R4:S    0.0
dtype: float64


In [311]:
severity_df.filter(like="PM8").max()

R1-PM8:V    0.0
R2-PM8:V    0.0
R3-PM8:V    0.0
R4-PM8:V    0.0
dtype: float64

In [276]:
severity_df.groupby(df["marker"]).mean().index

Index([27, 28, 29, 30, 35, 36, 37, 38, 39, 40, 41], dtype='int64', name='marker')

In [317]:
severity_df.groupby(df["marker"]).size().sort_index()

marker
27    11
28    24
29    26
30    86
35    82
36    47
37    43
38    53
39    48
40    16
41    64
dtype: int64

In [205]:
severity_df.filter(like="PM9").max()

R1-PM9:V    0.0
R2-PM9:V    0.0
R3-PM9:V    0.0
R4-PM9:V    0.0
dtype: float64

In [312]:
quiet_normal = (
    (df["marker"] == 41) &
    (df["R1:DF"].abs() < 0.05) &
    (df["R2:DF"].abs() < 0.05) &
    (df["R3:DF"].abs() < 0.05) &
    (df["R4:DF"].abs() < 0.05)
)

In [313]:
df_normal = df.loc[quiet_normal, FEATURES]

In [314]:
severity_df.filter(like="PM8").max()
severity_df.filter(like="PM9").max()
severity_df.filter(like="PA8").max()
severity_df.filter(like="PA9").max()

R1-PA9:VH    0.0
R2-PA9:VH    0.0
R3-PA9:VH    0.0
R4-PA9:VH    0.0
dtype: float64

In [315]:
severity_df.groupby(df["marker"]).mean().loc[[41, 1, 2, 39, 40]]

KeyError: '[1, 2] not in index'

In [346]:
# --------------------------------------------------
# Feature ownership map (STRICT physical ownership)
# --------------------------------------------------

ELEMENT_FEATURES = {

    # -------------------------
    # RELAYS
    # -------------------------
    "R1": [c for c in severity_df.columns if c.startswith("R1-")],
    "R2": [c for c in severity_df.columns if c.startswith("R2-")],
    "R3": [c for c in severity_df.columns if c.startswith("R3-")],
    "R4": [c for c in severity_df.columns if c.startswith("R4-")],

    # -------------------------
    # BREAKERS
    # -------------------------
    "BR1": [c for c in severity_df.columns
            if c.startswith("R1-") and (c.endswith(":S") or ":I" in c)],

    "BR2": [c for c in severity_df.columns
            if c.startswith("R2-") and (c.endswith(":S") or ":I" in c)],

    "BR3": [c for c in severity_df.columns
            if c.startswith("R3-") and (c.endswith(":S") or ":I" in c)],

    "BR4": [c for c in severity_df.columns
            if c.startswith("R4-") and (c.endswith(":S") or ":I" in c)],

    # -------------------------
    # TRANSMISSION LINES
    # -------------------------
    "L1": [c for c in severity_df.columns
           if (c.startswith("R1-") or c.startswith("R2-"))
           and (":I" in c or ":Z" in c or ":V" in c)],

    "L2": [c for c in severity_df.columns
           if (c.startswith("R3-") or c.startswith("R4-"))
           and (":I" in c or ":Z" in c or ":V" in c)],

    # -------------------------
    # GENERATORS
    # -------------------------
    "G1": [c for c in severity_df.columns
           if c.startswith("R1-")
           and (c.endswith(":V") or c.endswith(":F") or c.endswith(":DF") or ":I" in c)],

    "G2": [c for c in severity_df.columns
           if c.startswith("R4-")
           and (c.endswith(":V") or c.endswith(":F") or c.endswith(":DF") or ":I" in c)],
}

In [347]:
# --------------------------------------------------
# Element-level feature weights
# These weights reflect the physical importance of
# different measurements for each grid element.
# All weights sum to 1.0 per element.
# --------------------------------------------------

ELEMENT_WEIGHTS = {

    # --------------------------------------------------
    # RELAY
    # Protection logic is driven primarily by current
    # and impedance, not status alone.
    # Status is deliberately non-dominant to allow
    # detection of disabled-relay attacks.
    # --------------------------------------------------
    "relay": {
        "current_magnitude": 0.30,      # fault current detection
        "impedance": 0.25,              # distance protection
        "impedance_angle": 0.15,        # fault direction
        "current_phase_angle": 0.15,    # phase imbalance / fault type
        "rocof": 0.10,                  # instability indicator
        "status": 0.05                  # relay asserted / not asserted
    },

    # --------------------------------------------------
    # BREAKER
    # Binary device: open/close state dominates,
    # current confirms physical operation.
    # --------------------------------------------------
    "breaker": {
        "status": 1.0                # open / closed
        "current_magnitude": 0.30       # confirms current interruption
    },

    # --------------------------------------------------
    # TRANSMISSION LINE
    # Thermal and fault stress dominate.
    # --------------------------------------------------
    "line": {
        "current_magnitude": 0.40,      # thermal loading
        "impedance": 0.25,              # fault indication
        "voltage_magnitude": 0.20,      # voltage sag / swell
        "frequency": 0.10,              # system disturbance
        "voltage_phase_angle": 0.05     # power flow deviation
    },

    # --------------------------------------------------
    # GENERATOR
    # Frequency stability is primary.
    # --------------------------------------------------
    "generator": {
        "frequency": 0.35,              # speed / stability
        "rocof": 0.25,                  # loss of generation
        "voltage_magnitude": 0.20,      # excitation control
        "current_magnitude": 0.20       # electrical loading
    }
}

SyntaxError: invalid syntax (2803083689.py, line 33)

In [348]:
row_id = 324
elem = "R1"

owned = ELEMENT_FEATURES[elem]

# element type
etype = "relay"
weights = ELEMENT_WEIGHTS[etype]

print("Marker:", df.loc[row_id, "marker"])
print("Element:", elem, "etype:", etype)

for ftype, w in weights.items():
    cols = [c for c in owned if FEATURE_TYPES.get(c) == ftype and c in severity_df.columns]
    if not cols:
        continue

    maxv = severity_df.loc[row_id, cols].max()
    # show which feature(s) hit the max
    top = severity_df.loc[row_id, cols].sort_values(ascending=False).head(5)

    print(f"\n{ftype} weight={w}")
    print("max severity =", maxv)
    print("top features:")
    print(top)

Marker: 39
Element: R1 etype: relay

current_magnitude weight=0.3
max severity = 0.6
top features:
R1-PM4:I     0.6
R1-PM6:I     0.6
R1-PM11:I    0.6
R1-PM12:I    0.6
R1-PM5:I     0.0
Name: 324, dtype: float64

impedance weight=0.25
max severity = 0.0
top features:
R1-PA:Z    0.0
Name: 324, dtype: float64

impedance_angle weight=0.15
max severity = 0.0
top features:
R1-PA:ZH    0.0
Name: 324, dtype: float64

current_phase_angle weight=0.15
max severity = 0.0
top features:
R1-PA4:IH     0.0
R1-PA5:IH     0.0
R1-PA6:IH     0.0
R1-PA10:IH    0.0
R1-PA11:IH    0.0
Name: 324, dtype: float64


In [349]:
# --------------------------------------------------
# Aggregate feature severities to element-level score
# --------------------------------------------------

# --------------------------------------------------
# Element health computation (ownership + weights)
# --------------------------------------------------

def compute_element_health(severity_df, element_features, element_weights, feature_types):
    """
    Correct element health:
      health_E(t) = Σ_{type} weight(type) * max_severity_over_owned_features_of_that_type(t)

    severity_df: DataFrame [time x features], values in [0,1]
    element_features: dict {element -> [owned feature names]}
    element_weights: dict {element_type -> {feature_type: weight}}
    feature_types: dict {feature -> feature_type}
    """

    element_health = pd.DataFrame(index=severity_df.index)

    for element, owned in element_features.items():

        # ---- determine element type
        if element.startswith("BR"):
            etype = "breaker"
        elif element.startswith("R"):
            etype = "relay"
        elif element.startswith("L"):
            etype = "line"
        elif element.startswith("G"):
            etype = "generator"
        else:
            raise ValueError(f"Unknown element: {element}")

        weights = element_weights[etype]

        # start at 0 for all time
        score = pd.Series(0.0, index=severity_df.index)

        # apply each weight ONCE per type
        for ftype, w in weights.items():
            cols = [f for f in owned if feature_types.get(f) == ftype]
            if not cols:
                continue

            # max severity across owned features of this type
            type_max = severity_df[cols].max(axis=1)
            score += w * type_max

        element_health[element] = score

    return element_health



# ---- Run it on a SMALL slice first (fast sanity check)
df_small = severity_df.sample(n=500, random_state=42)   # <-- adjust size if needed

element_health = compute_element_health(
    df_small,
    ELEMENT_FEATURES,
    ELEMENT_WEIGHTS,
    FEATURE_TYPES
)

print("Element health shape:", element_health.shape)
element_health.head(50)
element_health_with_marker = element_health.copy()
element_health_with_marker["marker"] = df.loc[element_health.index, "marker"]

element_health_with_marker.head(50)

Element health shape: (500, 12)


,R1,R2,R3,R4,BR1,BR2,BR3,BR4,L1,L2,G1,G2,marker
38854,0.00,0.15,0.15,0.00,0.00,0.00,0.00,0.00,0.12,0.12,0.00,0.00,24
74030,0.60,0.69,0.40,0.00,0.30,0.30,0.00,0.00,0.75,0.45,0.40,0.20,36
60133,0.00,0.24,0.24,0.00,0.00,0.00,0.00,0.00,0.12,0.12,0.00,0.12,24
36602,0.27,0.33,0.33,0.27,0.18,0.18,0.18,0.18,0.36,0.36,0.24,0.24,41
64691,0.27,0.33,0.33,0.27,0.18,0.18,0.18,0.18,0.36,0.36,0.24,0.12,28
701,0.09,0.15,0.24,0.09,0.00,0.00,0.00,0.00,0.03,0.03,0.00,0.00,37
43587,0.18,0.33,0.33,0.18,0.18,0.18,0.18,0.18,0.36,0.36,0.24,0.24,26
55511,0.00,0.15,0.15,0.00,0.00,0.00,0.00,0.00,0.12,0.12,0.12,0.12,19
11306,0.00,0.24,0.24,0.00,0.00,0.00,0.00,0.00,0.12,0.12,0.12,0.12,36
19663,0.09,0.15,0.15,0.00,0.00,0.00,0.00,0.00,0.12,0.12,0.12,0.12,10


In [242]:
fault_idx = df[df["marker"].isin([1,2,3,4,5,6])].index[0]
element_health.loc[fault_idx]

KeyError: np.int64(4126)

In [182]:
# Pick ONLY normal-operation rows
normal_mask = df["marker"] == 41
baseline = element_health.loc[normal_mask].median()
element_delta = element_health.subtract(baseline)
element_delta_with_marker = element_delta.copy()
element_delta_with_marker["marker"] = df.loc[element_delta.index, "marker"]

element_delta_with_marker.query("marker != 41").head()
# # How many non-zero severities exist in normal?
# normal_nonzero = (severity_df[normal_mask] > 0).mean().sort_values(ascending=False)

# print("Top 20 features causing non-zero severity in NORMAL (marker=41):")
# display(normal_nonzero.head(40))

# print("\nOverall fraction of non-zero severities in NORMAL:",
#       (severity_df[normal_mask] > 0).values.mean())

,R1,R2,R3,R4,BR1,BR2,BR3,BR4,L1,L2,G1,G2,marker
1242,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,35
307,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40
1333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,35
993,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,36
514,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,38


In [74]:
# --------------------------------------------------
# Compute element health time series
# --------------------------------------------------

ELEMENTS = {
    "R1": "relay", "R2": "relay", "R3": "relay", "R4": "relay",
    "BR1": "breaker", "BR2": "breaker", "BR3": "breaker", "BR4": "breaker",
    "L1": "line", "L2": "line",
    "G1": "generator", "G2": "generator"
}

# Randomly sample 200 timesteps
severity_small = severity_df.sample(n=50, random_state=42)
element_health_small[elem] = severity_small.apply(
    lambda row: aggregate_element_severity(
        severity_row=row,
        element=elem,          # 👈 ADD THIS
        element_type=etype,
        feature_types=FEATURE_TYPES
    ),
    axis=1
)
element_health_small.sort_index()
# --------------------------------------------------
# Combine element health with scenario marker
# --------------------------------------------------

element_health_with_marker = element_health_small.copy()
element_health_with_marker["marker"] = df.loc[element_health.index, "marker"]

element_health_with_marker.head(50)
    

,R1,R2,R3,R4,BR1,BR2,BR3,BR4,L1,L2,G1,G2,marker
38854,0.30,0.45,0.45,0.30,0.3,0.3,0.3,0.3,0.40,0.40,0.2,0.2,24
74030,0.60,0.60,0.70,0.30,0.3,0.3,0.3,0.3,0.75,0.85,0.4,0.4,36
60133,0.30,0.45,0.45,0.30,0.3,0.3,0.3,0.3,0.40,0.40,0.2,0.2,24
36602,0.30,0.45,0.45,0.30,0.3,0.3,0.3,0.3,0.40,0.40,0.2,0.2,41
64691,0.30,0.45,0.45,0.30,0.3,0.3,0.3,0.3,0.60,0.60,0.2,0.2,28
701,0.30,0.45,0.45,0.30,0.3,0.3,0.3,0.3,0.40,0.40,0.2,0.2,37
43587,0.30,0.45,0.45,0.30,0.3,0.3,0.3,0.3,0.60,0.60,0.2,0.2,26
55511,0.30,0.45,0.45,0.30,0.3,0.3,0.3,0.3,0.40,0.40,0.2,0.2,19
11306,0.30,0.45,0.45,0.30,0.3,0.3,0.3,0.3,0.40,0.40,0.2,0.2,36
19663,0.30,0.45,0.45,0.30,0.3,0.3,0.3,0.3,0.40,0.40,0.2,0.2,10


In [77]:
normal_idx = df.index[df["marker"] == 41]
element_health.loc[normal_idx].max()

KeyError: '[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220

In [76]:
element_health[df["marker"].isin([7,8,9,10,11,12])].max()

/var/folders/nv/f9_p0fkx4xsf4ygmqb8ynm9w0000gn/T/ipykernel_79426/4243072325.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  element_health[df["marker"].isin([7,8,9,10,11,12])].max()


R1     0.75
R2     0.75
R3     0.75
R4     0.75
BR1    1.00
BR2    1.00
BR3    1.00
BR4    1.00
L1     0.77
L2     0.77
G1     0.40
G2     0.40
dtype: float64

In [66]:
severity_small = severity_df.sample(1000, random_state=42)
element_health = pd.DataFrame(index=severity_small.index)

for elem, etype in ELEMENTS.items():
    element_health[elem] = severity_small.apply(
        lambda row: aggregate_element_severity(row, etype, FEATURE_TYPES),
        axis=1
    )
element_health


    

,R1,R2,R3,R4,BR1,BR2,BR3,BR4,L1,L2,G1,G2,marker
38854,0.50,0.50,0.50,0.50,1.0,1.0,1.0,1.0,0.40,0.40,0.2,0.2,24
74030,0.75,0.75,0.75,0.75,1.0,1.0,1.0,1.0,0.85,0.85,0.4,0.4,36
60133,0.50,0.50,0.50,0.50,1.0,1.0,1.0,1.0,0.40,0.40,0.2,0.2,24
36602,0.50,0.50,0.50,0.50,1.0,1.0,1.0,1.0,0.40,0.40,0.2,0.2,41
64691,0.50,0.50,0.50,0.50,1.0,1.0,1.0,1.0,0.60,0.60,0.4,0.4,28
701,0.50,0.50,0.50,0.50,1.0,1.0,1.0,1.0,0.40,0.40,0.2,0.2,37
43587,0.50,0.50,0.50,0.50,1.0,1.0,1.0,1.0,0.60,0.60,0.4,0.4,26
55511,0.50,0.50,0.50,0.50,1.0,1.0,1.0,1.0,0.40,0.40,0.2,0.2,19
11306,0.50,0.50,0.50,0.50,1.0,1.0,1.0,1.0,0.40,0.40,0.2,0.2,36
19663,0.50,0.50,0.50,0.50,1.0,1.0,1.0,1.0,0.40,0.40,0.2,0.2,10


In [60]:
t = 1000
row = severity_df.iloc[t]

for elem, etype in ELEMENTS.items():
    print(elem, aggregate_element_severity(row, etype, FEATURE_TYPES))

R1 0.49999999999999994
R2 0.49999999999999994
R3 0.49999999999999994
R4 0.49999999999999994
BR1 1.0
BR2 1.0
BR3 1.0
BR4 1.0
L1 0.4
L2 0.4
G1 0.2
G2 0.2


In [431]:
sample = bounds_df["median"].copy()
sample["R3-PM12:I"] = bounds_df.loc["R3-PM12:I", "q75"] + 5 * (
    bounds_df.loc["R3-PM12:I", "q75"] - bounds_df.loc["R3-PM12:I", "q25"]
)

states = evaluate_feature_states(sample, bounds_df)
print(states["R3-PM12:I"])

GREEN


In [432]:
print(bounds_df.loc["R3-PM12:I", "type"])

current_magnitude


In [452]:
from collections import Counter

print("="*80)
print("TEST 1 — NORMAL BASELINE (MEDIAN VALUES)")
print("="*80)

sample = bounds_df["median"].copy()
states = evaluate_feature_states(sample, bounds_df)

counts = Counter(states.values())
print("State counts:", counts)

# Detailed print (optional but useful)
for f, s in list(states.items())[:20]:
    print(f"{f:25s} -> {s}")

assert all(v == "GREEN" for v in states.values())
print("\n✅ PASS: All median values correctly classified as GREEN")

TEST 1 — NORMAL BASELINE (MEDIAN VALUES)
State counts: Counter({'GREEN': 132})
R1-PA1:VH                 -> GREEN
R1-PM1:V                  -> GREEN
R1-PA2:VH                 -> GREEN
R1-PM2:V                  -> GREEN
R1-PA3:VH                 -> GREEN
R1-PM3:V                  -> GREEN
R1-PA4:IH                 -> GREEN
R1-PM4:I                  -> GREEN
R1-PA5:IH                 -> GREEN
R1-PM5:I                  -> GREEN
R1-PA6:IH                 -> GREEN
R1-PM6:I                  -> GREEN
R1-PA7:VH                 -> GREEN
R1-PM7:V                  -> GREEN
R1-PA8:VH                 -> GREEN
R1-PM8:V                  -> GREEN
R1-PA9:VH                 -> GREEN
R1-PM9:V                  -> GREEN
R1-PA10:IH                -> GREEN
R1-PM10:I                 -> GREEN

✅ PASS: All median values correctly classified as GREEN


In [453]:
print("\n" + "="*80)
print("TEST 2 — FEATURE TYPE ABNORMALITY CHECK")
print("="*80)

def test_feature_type(feature_type, multiplier=2.0):
    features = bounds_df[bounds_df["type"] == feature_type].index
    if len(features) == 0:
        print(f"[{feature_type}] — no features found")
        return

    f = features[0]
    row = bounds_df.loc[f]

    sample = bounds_df["median"].copy()

    # Inject abnormal value
    if feature_type in ["current_magnitude", "voltage_magnitude"]:
        sample[f] = row["p99"] * multiplier
    elif feature_type in ["phase_angle", "impedance_angle", "rocof"]:
        sample[f] = row["median"] + 4 * max(row["p95"] - row["median"], 1e-6)
    elif feature_type == "impedance":
        sample[f] = max(row["q25"] - 4 * (row["q75"] - row["q25"]), 0)
    elif feature_type == "frequency":
        sample[f] = row["median"] + 1.0
    elif feature_type == "status":
        sample[f] = row["median"] + 1
    else:
        print(f"[{feature_type}] skipped (non-physical)")
        return

    state = feature_state(sample[f], row)

    print(f"\nFeature: {f}")
    print(f"Type: {feature_type}")
    print(f"Injected value: {sample[f]}")
    print(f"Median: {row['median']}")
    print(f"p99: {row.get('p99', 'N/A')}")
    print(f"Resulting state: {state}")

# Run tests for all physical types
for t in [
    "voltage_magnitude",
    "current_magnitude",
    "phase_angle",
    "impedance",
    "impedance_angle",
    "frequency",
    "rocof",
    "status"
]:
    test_feature_type(t)


TEST 2 — FEATURE TYPE ABNORMALITY CHECK

Feature: R1-PM1:V
Type: voltage_magnitude
Injected value: 266428.567
Median: 131659.7408
p99: 133214.2835
Resulting state: RED

Feature: R1-PM4:I
Type: current_magnitude
Injected value: 1130.8580623999999
Median: 393.86961
p99: 565.4290311999999
Resulting state: RED

Feature: R1-PA1:VH
Type: phase_angle
Injected value: 596.5900104000001
Median: -3.443476
p99: 177.30728984
Resulting state: RED

Feature: R1-PA:Z
Type: impedance
Injected value: 0
Median: 9.598383018
p99: 14.826951748399999
Resulting state: AMBER

Feature: R1-PA:ZH
Type: impedance_angle
Injected value: 0.2558122
Median: 0.036937
p99: 0.09997512
Resulting state: RED

Feature: R1:F
Type: frequency
Injected value: 61.0
Median: 60.0
p99: 60.0015
Resulting state: RED

Feature: R1:DF
Type: rocof
Injected value: 4e-06
Median: 0.0
p99: 0.0
Resulting state: RED

Feature: R1:S
Type: status
Injected value: 1.0
Median: 0.0
p99: 0.0
Resulting state: RED


In [460]:
# ============================================================
# NOTEBOOK CELL: Comprehensive "feature_state" test suite
# Tests ALL conditions across ALL feature types + prints results
# ============================================================

import numpy as np
import pandas as pd
from collections import Counter, defaultdict

# ------------------------------------------------------------
# 0) Safety: make sure these exist from earlier cells
#    - bounds_df : DataFrame indexed by feature name
#    - feature_state(value, row) : your function
# ------------------------------------------------------------
assert isinstance(bounds_df, pd.DataFrame)
assert "type" in bounds_df.columns

# ------------------------------------------------------------
# 1) Optional: wrap feature_state with NaN/Inf handling
#    (Recommended for runtime robustness)
# ------------------------------------------------------------
def feature_state_safe(value, row):
    # For any non-finite runtime value, mark RED immediately
    # (measurement issue / invalid physics)
    if value is None or (isinstance(value, float) and not np.isfinite(value)) or (isinstance(value, (int, np.integer)) is False and hasattr(value, "__float__") and not np.isfinite(float(value))):
        return "RED"
    if isinstance(value, (np.floating, float)) and not np.isfinite(value):
        return "RED"
    return feature_state(value, row)


# ------------------------------------------------------------
# 2) Utility: choose a "representative" feature per type
# ------------------------------------------------------------
def pick_feature_per_type(bounds_df):
    reps = {}
    for t, g in bounds_df.groupby("type"):
        reps[t] = g.index[0]
    return reps

reps = pick_feature_per_type(bounds_df)
print("Representative feature per type:")
for t, f in reps.items():
    print(f"  {t:16s} -> {f}")

# ------------------------------------------------------------
# 3) Injection recipes per type
#    We target each IF-branch in your feature_state logic:
#      - GREEN band
#      - AMBER band
#      - RED band
#    plus degenerate cases (IQR=0) and p95/p99 thresholds.
# ------------------------------------------------------------
def inject_values_for_feature(row):
    """
    Returns a dict of named test values that should hit different conditions.
    Works even when IQR collapses (median=q25=q75).
    """
    t = row["type"]
    median = float(row["median"])
    q25 = float(row["q25"])
    q75 = float(row["q75"])
    p95 = float(row.get("p95", q75))
    p99 = float(row.get("p99", q75))
    iqr = q75 - q25

    # fallback spread (same idea as your code)
    if iqr == 0:
        spread = max(abs(p95 - median), abs(p99 - median), 1e-6)
    else:
        spread = iqr

    tests = {}

    # universal sanity
    tests["GREEN@median"] = median
    tests["RED@nan"] = np.nan
    tests["RED@inf"] = np.inf
    tests["RED@-inf"] = -np.inf

    # Voltage magnitude: RED if outside q25-3*spread or q75+3*spread
    if t == "voltage_magnitude":
        tests["GREEN@within"] = median
        tests["AMBER@high"]  = q75 + 1.1*spread
        tests["AMBER@low"]   = q25 - 1.1*spread
        tests["RED@high"]    = q75 + 3.1*spread
        tests["RED@low"]     = q25 - 3.1*spread

    # Current magnitude:
    # - if degenerate iqr==0: AMBER if >=p95, RED if >=p99
    # - else: AMBER if >q75+spread, RED if >q75+3*spread
    elif t == "current_magnitude":
        tests["GREEN@within"] = median
        if iqr == 0:
            tests["AMBER@p95"] = p95
            tests["RED@p99"]   = p99
            tests["RED@2xp99"] = p99 * 2.0
        else:
            tests["AMBER@high"] = q75 + 1.1*spread
            tests["RED@high"]   = q75 + 3.1*spread

    # Phase / impedance angles / rocof: compare deviation vs spread
    elif t in ["phase_angle", "impedance_angle", "rocof"]:
        tests["GREEN@within"] = median
        tests["AMBER@dev"]    = median + 1.1*spread
        tests["RED@dev"]      = median + 3.1*spread

    # Impedance magnitude: RED if value < q25 - 3*spread; AMBER if < q25 - spread
    elif t == "impedance":
        tests["GREEN@within"] = median
        tests["AMBER@low"]    = q25 - 1.1*spread
        tests["RED@low"]      = q25 - 3.1*spread

    # Frequency: hard thresholds
    elif t == "frequency":
        tests["GREEN@within"] = median
        tests["AMBER@0.3Hz"]  = median + 0.3
        tests["RED@0.6Hz"]    = median + 0.6

    # Status: exact equality
    elif t == "status":
        tests["GREEN@equal"]  = median
        tests["RED@flip"]     = median + 1 if median == 0 else median - 1

    # measurement_flag / cyber_log / other (if you still keep them):
    # your function returns GREEN for these (physics ignores)
    else:
        tests["GREEN@median"] = median
        tests["GREEN@weird"]  = median + 9999

    return tests


# ------------------------------------------------------------
# 4) Run tests for ONE representative feature per type (fast)
# ------------------------------------------------------------
print("\n" + "="*80)
print("TEST A — Representative feature per type (hits all branches)")
print("="*80)

rep_results = []
for t, feat in reps.items():
    row = bounds_df.loc[feat]
    tests = inject_values_for_feature(row)

    for test_name, injected in tests.items():
        st = feature_state_safe(injected, row)
        rep_results.append((t, feat, test_name, injected, st))

rep_df = pd.DataFrame(rep_results, columns=["type", "feature", "test", "injected_value", "state"])
print(rep_df.to_string(index=False))

print("\nState counts (representatives):", Counter(rep_df["state"]))


# ------------------------------------------------------------
# 5) Run tests for EVERY feature (more thorough, still manageable)
#    We only inject a small set per feature to avoid huge outputs.
# ------------------------------------------------------------
print("\n" + "="*80)
print("TEST B — Full sweep: every feature (summary only)")
print("="*80)

# For each feature, we test a compact set: median + two "abnormal" values.
# (Rules depend on type, so we generate from inject_values_for_feature and pick key ones.)
def compact_tests(row):
    tests = inject_values_for_feature(row)
    # keep a consistent subset
    keep = ["GREEN@median", "AMBER@high", "AMBER@low", "AMBER@dev", "AMBER@p95", "RED@high", "RED@low", "RED@dev", "RED@p99"]
    out = {"GREEN@median": tests["GREEN@median"]}
    for k in keep[1:]:
        if k in tests:
            out[k] = tests[k]
    # always include NaN/Inf checks
    out["RED@nan"] = tests["RED@nan"]
    out["RED@inf"] = tests["RED@inf"]
    return out

full_counts_by_type = defaultdict(Counter)
examples_bad = []

for feat in bounds_df.index:
    row = bounds_df.loc[feat]
    t = row["type"]
    tests = compact_tests(row)

    for test_name, injected in tests.items():
        st = feature_state_safe(injected, row)
        full_counts_by_type[t][st] += 1

        # store a few surprising cases for inspection
        if test_name.startswith("RED@") and st != "RED":
            if len(examples_bad) < 20:
                examples_bad.append((feat, t, test_name, injected, st, float(row["median"]), float(row["q25"]), float(row["q75"]), float(row.get("p99", row["q75"]))))

print("State totals per type (across all features and compact tests):")
for t in sorted(full_counts_by_type.keys()):
    print(f"  {t:16s} -> {dict(full_counts_by_type[t])}")

if examples_bad:
    print("\n⚠️ Unexpected cases (expected RED but got AMBER/GREEN) — showing up to 20:")
    bad_df = pd.DataFrame(
        examples_bad,
        columns=["feature","type","test","injected","state","median","q25","q75","p99"]
    )
    print(bad_df.to_string(index=False))
else:
    print("\n✅ No unexpected cases where a RED injection failed to produce RED.")


# ------------------------------------------------------------
# 6) Test C — "All medians must be GREEN" (your sanity test)
# ------------------------------------------------------------
print("\n" + "="*80)
print("TEST C — All medians GREEN")
print("="*80)

median_sample = bounds_df["median"].copy()
median_states = {f: feature_state_safe(median_sample[f], bounds_df.loc[f]) for f in bounds_df.index}
median_counter = Counter(median_states.values())
print("Median state counts:", median_counter)

if all(v == "GREEN" for v in median_states.values()):
    print("✅ PASS: Every feature is GREEN at its median.")
else:
    # print first few offenders
    offenders = [f for f, st in median_states.items() if st != "GREEN"][:20]
    print("❌ FAIL: Some medians are not GREEN. First offenders:", offenders)


# ------------------------------------------------------------
# 7) Test D — Build a "printable table" of bounds per type
#    (This gives you the table you asked for: feature type bounds)
# ------------------------------------------------------------
print("\n" + "="*80)
print("TEST D — Bounds table per feature type (min/median/max over features)")
print("="*80)

# summarize bounds_df by type: for each statistic column, take min/median/max across features of that type
stats_cols = ["median","q25","q75","p05","p95","p99"]
summary_rows = []
for t, g in bounds_df.groupby("type"):
    row = {"type": t, "n_features": len(g)}
    for c in stats_cols:
        if c in g.columns:
            row[f"{c}_min"] = float(g[c].min())
            row[f"{c}_median"] = float(g[c].median())
            row[f"{c}_max"] = float(g[c].max())
    # IQR across features (note: this is across features, not within feature)
    if "q25" in g.columns and "q75" in g.columns:
        iqr_series = g["q75"] - g["q25"]
        row["IQR_min"] = float(iqr_series.min())
        row["IQR_median"] = float(iqr_series.median())
        row["IQR_max"] = float(iqr_series.max())
        row["IQR_zero_count"] = int((iqr_series == 0).sum())
    summary_rows.append(row)

type_bounds_table = pd.DataFrame(summary_rows).sort_values("type")
print(type_bounds_table.to_string(index=False))


# ------------------------------------------------------------
# 8) Optional: save these tables for dissertation appendix
# ------------------------------------------------------------
# rep_df.to_csv("TEST_representative_feature_state_results.csv", index=False)
# type_bounds_table.to_csv("BOUNDS_summary_by_type.csv", index=False)

print("\nDone.")

Representative feature per type:
  current_magnitude -> R1-PM4:I
  cyber_log        -> control_panel_log1
  frequency        -> R1:F
  impedance        -> R1-PA:Z
  impedance_angle  -> R1-PA:ZH
  measurement_flag -> R1-PA:Z_inf_flag
  phase_angle      -> R1-PA1:VH
  rocof            -> R1:DF
  status           -> R1:S
  voltage_magnitude -> R1-PM1:V

TEST A — Representative feature per type (hits all branches)
             type            feature         test  injected_value state
current_magnitude           R1-PM4:I GREEN@median    3.938696e+02 GREEN
current_magnitude           R1-PM4:I      RED@nan             NaN   RED
current_magnitude           R1-PM4:I      RED@inf             inf   RED
current_magnitude           R1-PM4:I     RED@-inf            -inf   RED
current_magnitude           R1-PM4:I GREEN@within    3.938696e+02 GREEN
current_magnitude           R1-PM4:I   AMBER@high    6.044644e+02 AMBER
current_magnitude           R1-PM4:I     RED@high    8.758334e+02   RED
        cy

In [425]:
ELEMENT_MAP = {

    "R1": [c for c in bounds_df.index
           if c.startswith("R1-") and (":Z" in c or ":I" in c or c.endswith(":S"))
           and "Z_inf_flag" not in c] + ["relay1_log"],

    "R2": [c for c in bounds_df.index
           if c.startswith("R2-") and (":Z" in c or ":I" in c or c.endswith(":S"))
           and "Z_inf_flag" not in c] + ["relay2_log"],

    "R3": [c for c in bounds_df.index
           if c.startswith("R3-") and (":Z" in c or ":I" in c or c.endswith(":S"))
           and "Z_inf_flag" not in c] + ["relay3_log"],

    "R4": [c for c in bounds_df.index
           if c.startswith("R4-") and (":Z" in c or ":I" in c or c.endswith(":S"))
           and "Z_inf_flag" not in c] + ["relay4_log"],

    "BR1": ["R1:S"],
    "BR2": ["R2:S"],
    "BR3": ["R3:S"],
    "BR4": ["R4:S"],

    "B1": [c for c in bounds_df.index if c.startswith("R1-") and ":V" in c],
    "B2": [c for c in bounds_df.index if c.startswith("R2-") and ":V" in c],
    "B3": [c for c in bounds_df.index if c.startswith("R4-") and ":V" in c],

    "G1": [c for c in bounds_df.index if c.startswith("R1-")
           and (":V" in c or c.endswith(":F") or c.endswith(":DF"))],

    "G2": [c for c in bounds_df.index if c.startswith("R4-")
           and (":V" in c or c.endswith(":F") or c.endswith(":DF"))],
}

LINE_MAP = {
    "L1": {
        "current_features": [c for c in bounds_df.index
                             if (c.startswith("R1-") or c.startswith("R2-")) and ":I" in c],
        "breaker_status": ["R1:S", "R2:S"]
    },
    "L2": {
        "current_features": [c for c in bounds_df.index
                             if (c.startswith("R3-") or c.startswith("R4-")) and ":I" in c],
        "breaker_status": ["R3:S", "R4:S"]
    }
}

In [442]:
# =========================
# TEST 1: Normal behaviour
# =========================

normal_idx = df[df["marker"] == 41].index[0]
sample = df.loc[normal_idx].reindex(FEATURES)

feature_states = evaluate_feature_states(sample, bounds_df)

from collections import Counter
print("Normal feature state counts:")
print(Counter(feature_states.values()))

Normal feature state counts:
Counter({'GREEN': 111, 'AMBER': 19, 'RED': 2})


In [443]:
# =========================
# TEST 2: Fault current
# =========================

fault_idx = df[df["marker"].isin([1,2,3,4,5,6])].index[0]
sample = df.loc[fault_idx].reindex(FEATURES)

feature_states = evaluate_feature_states(sample, bounds_df)

# Show only current features
for f, state in feature_states.items():
    if FEATURE_TYPES[f] == "current_magnitude":
        print(f"{f:25s} -> {state}")

R1-PM4:I                  -> GREEN
R1-PM5:I                  -> GREEN
R1-PM6:I                  -> GREEN
R1-PM10:I                 -> GREEN
R1-PM11:I                 -> GREEN
R1-PM12:I                 -> GREEN
R2-PM4:I                  -> GREEN
R2-PM5:I                  -> GREEN
R2-PM6:I                  -> GREEN
R2-PM10:I                 -> GREEN
R2-PM11:I                 -> GREEN
R2-PM12:I                 -> GREEN
R3-PM4:I                  -> GREEN
R3-PM5:I                  -> GREEN
R3-PM6:I                  -> GREEN
R3-PM10:I                 -> GREEN
R3-PM11:I                 -> GREEN
R3-PM12:I                 -> GREEN
R4-PM4:I                  -> GREEN
R4-PM5:I                  -> GREEN
R4-PM6:I                  -> GREEN
R4-PM10:I                 -> GREEN
R4-PM11:I                 -> GREEN
R4-PM12:I                 -> GREEN


In [426]:
def detect_line_maintenance(sample, line_id):
    """
    Dataset-faithful detection of line maintenance (BLUE)
    Scenarios 13–14 only
    """

    line = LINE_MAP[line_id]

    # 1️⃣ Breaker opened
    breakers_open = any(
        sample.get(b, 1) == 0
        for b in line["breaker_status"]
    )

    # 2️⃣ Line current near zero (relative to normal)
    currents = [
        abs(sample[c]) for c in line["current_features"]
        if c in sample
    ]
    if not currents:
        return None

    median_currents = [
        bounds_df.loc[c, "median"]
        for c in line["current_features"]
        if c in bounds_df.index
    ]

    low_current = max(currents) < 0.15 * max(median_currents)

    # 3️⃣ Voltages normal (no fault)
    voltage_features = [
        c for c in bounds_df.index
        if (
            (line_id == "L1" and c.startswith(("R1-", "R2-"))) or
            (line_id == "L2" and c.startswith(("R3-", "R4-")))
        ) and ":V" in c
    ]

    voltages_ok = all(
        feature_state(sample[v], bounds_df.loc[v]) == "GREEN"
        for v in voltage_features
        if v in sample
    )

    # 4️⃣ No cyber activity
    no_cyber = all(
        sample.get(f"snort_log{i}", 0) == 0
        for i in range(1, 5)
    )

    if breakers_open and low_current and voltages_ok and no_cyber:
        return "BLUE"

    return None

In [444]:
sample = bounds_df["median"].copy()

f = "R3-PM12:I"
row = bounds_df.loc[f]

# Inject a real fault-level current
sample[f] = row["p99"] * 2   # or *3

print("Injected:", sample[f])
print("p99:", row["p99"])
print(feature_state(sample[f], row))

Injected: 33.87535
p99: 16.937675
AMBER


In [415]:
IDX = 8593   # change freely
sample = df.loc[IDX].reindex(FEATURES)

feature_states = evaluate_feature_states(sample, bounds_df)

In [413]:
maint = df[df["marker"].isin([13, 14])]

for idx in maint.index[:10]:
    sample = df.loc[idx].reindex(FEATURES)
    print(idx,
          detect_line_maintenance(sample, "L1"),
          detect_line_maintenance(sample, "L2"))

3177 None None
3178 None None
3179 None None
3180 None None
3181 None None
3182 None None
3183 None None
3184 None None
3185 None None
3186 None None


In [404]:
def element_state(element_id, feature_states):
    """
    Aggregate feature states into an element state
    """

    features = ELEMENT_MAP[element_id]
    states = [
        feature_states[f]
        for f in features
        if f in feature_states
    ]

    if not states:
        return "GREEN"

    red_count = states.count("RED")
    amber_count = states.count("AMBER")

    # Conservative logic
    if red_count >= 2:
        return "RED"
    if red_count == 1 or amber_count >= 2:
        return "AMBER"

    return "GREEN"

In [405]:
def compute_grid_states(sample, feature_states):

    line_states = {
        line: detect_line_maintenance(sample, line)
        for line in LINE_MAP
    }

    maintained = [l for l, s in line_states.items() if s == "BLUE"]

    # Remove maintained-line features
    active_features = {
        f: s for f, s in feature_states.items()
        if not (
            ("L1" in maintained and f.startswith(("R1-", "R2-"))) or
            ("L2" in maintained and f.startswith(("R3-", "R4-")))
        )
    }

    grid = {}

    for line in LINE_MAP:
        grid[line] = "BLUE" if line in maintained else "GREEN"

    for el in ELEMENT_MAP:
        grid[el] = element_state(el, active_features)

    return grid

In [406]:
IDX = 8593   # try different scenarios

sample = df.loc[IDX].reindex(FEATURES)
feature_states = evaluate_feature_states(sample, bounds_df)
grid_states = compute_grid_states(sample, feature_states)

print("\n=== GRID STATES ===")
for k in sorted(grid_states):
    print(f"{k:4s} -> {grid_states[k]}")


=== GRID STATES ===
B1   -> GREEN
B2   -> AMBER
B3   -> GREEN
BR1  -> GREEN
BR2  -> GREEN
BR3  -> GREEN
BR4  -> GREEN
G1   -> GREEN
G2   -> GREEN
L1   -> GREEN
L2   -> GREEN
R1   -> AMBER
R2   -> GREEN
R3   -> RED
R4   -> AMBER


In [418]:
sample = df[df["marker"] == 41].iloc[0].reindex(FEATURES)

# Inject an actual fault-level current
sample["R3-PM12:I"] = bounds_df.loc["R3-PM12:I", "q75"] * 10

states = evaluate_feature_states(sample, bounds_df)
print(states["R3-PM12:I"])

GREEN


In [247]:
sample = df.loc[7893]
# 2. Evaluate feature states
feature_states = evaluate_feature_states(sample, bounds_df)

# 3. Aggregate to grid
grid_states = {}

for element, feature_list in ELEMENT_MAP.items():
    maintenance_override = detect_maintenance(feature_states, element)

    if maintenance_override:
        grid_states[element] = maintenance_override
    else:
        grid_states[element] = element_state(feature_states, feature_list)

grid_states

{'R1': 'RED',
 'R2': 'RED',
 'R3': 'RED',
 'R4': 'RED',
 'BR1': 'GREEN',
 'BR2': 'GREEN',
 'BR3': 'GREEN',
 'BR4': 'GREEN',
 'B1': 'GREEN',
 'B2': 'GREEN',
 'B3': 'GREEN',
 'G1': 'GREEN',
 'G2': 'GREEN'}

In [226]:
# Create a synthetic perfect-normal sample
sample_green = bounds_df["median"].copy()

# Evaluate feature states
feature_states = {
    col: feature_state(sample_green[col], bounds_df.loc[col])
    for col in bounds_df.index
}

# Aggregate to grid
grid_states = {
    element: element_state(element, feature_states)
    for element in ELEMENT_MAP
}

grid_states

{'R1': 'GREEN',
 'R2': 'GREEN',
 'R3': 'GREEN',
 'R4': 'GREEN',
 'BR1': 'GREEN',
 'BR2': 'GREEN',
 'BR3': 'GREEN',
 'BR4': 'GREEN',
 'B1': 'GREEN',
 'B2': 'GREEN',
 'B3': 'GREEN',
 'G1': 'GREEN',
 'G2': 'GREEN'}

In [151]:
sample["R1-PM4:I"] = 0
sample["R2-PM4:I"] = 0
# 2. Evaluate feature states
feature_states = evaluate_feature_states(sample, bounds_df)

# 3. Aggregate to grid
grid_states = {}

for element, feature_list in ELEMENT_MAP.items():
    maintenance_override = detect_maintenance(feature_states, element)

    if maintenance_override:
        grid_states[element] = maintenance_override
    else:
        grid_states[element] = element_state(feature_states, feature_list)

grid_states

TypeError: 'int' object does not support item assignment

In [152]:
# Pick ONE row as a pandas Series
sample = df_normal.iloc[0].copy()   # <-- .copy() is critical

# Align to runtime feature order
sample = sample.reindex(FEATURES)

In [153]:
type(sample)

pandas.core.series.Series

In [154]:
# Simulate line open / maintenance
sample["R1-PM4:I"] = 0
sample["R2-PM4:I"] = 0

In [309]:
# === SANITY CHECK CELL ===

IDX = 8593  # known line maintenance
sample = df.loc[IDX].reindex(FEATURES)

feature_states = evaluate_feature_states(sample, bounds_df)

line_states = {
    line_id: detect_line_maintenance(sample, line_id)
    for line_id in LINE_MAP
}

grid_states = compute_grid_states(sample, feature_states)

print("Line states:", line_states)
print("Grid states:")
for k in sorted(grid_states):
    print(f"{k:4s} -> {grid_states[k]}")

KeyError: 'L1'

KeyError: 'L1'